## Legacy Model-Free Analysis Pipeline

In [ ]:
from pathlib import Path
from organize_simulated_fixations import reformat_fixations
from mfa import *

ROOT = Path("/Users/braydenchien/Desktop/Enkavilab/DDM")
SIM_DIR = ROOT / "simulated_data"
FORMATTED_DIR = ROOT / "formatted_data"
FORMATTED_DIR.mkdir(parents=True, exist_ok=True)

# Small formating function
def _fmt(x):
    # prints 1.5 not 1.500000, prints ints as ints if applicable
    return f"{x:g}"

bin_size = 0.001
seed = 42
model_conditions = {'drift_rate': 1.2, 'theta': 0.38, 'noise': 0.5}

base_name = f"sim_trials_s{seed}_d{_fmt(model_conditions['drift_rate'])}_t{_fmt(model_conditions['theta'])}_n{_fmt(model_conditions['noise'])}.csv"

reformat_fixations(str(SIM_DIR), base_name)

# 2) Run the four poster-figure saves (SVGs land in mfa_figures/)
formatted_parent_dir = str(FORMATTED_DIR)
formatted_data_path = base_name  # relative to formatted_parent_dir

save_choice_patterns(formatted_parent_dir, formatted_data_path)
save_response_dynamics(formatted_parent_dir, formatted_data_path)
save_fixation_structure(formatted_parent_dir, formatted_data_path)
save_value_modulated_fixation(formatted_parent_dir, formatted_data_path)

# print("Saved SVGs to ./mfa_figures for:")
# print(" - choice_patterns_*.svg")
# print(" - response_dynamics_*.svg")
# print(" - fixation_structure_*.svg")
# print(" - value_modulated_fixation_*.svg")

In [ ]:
# save_fixation_properties(formatted_parent_dir, formatted_data_path)
# save_basic_psychometrics(formatted_parent_dir, formatted_data_path)

## Local Batch Simulation

The following was legacy code used to generate the many csv files in ./simulated_data, ./formatted_data, and ./mfa_figures. If it is needed again, bin_size --> dt needs to be included in calls to get_empirical_distributions and create_trials.

In [ ]:
from simulation import get_empirical_distributions, create_model, create_trials, simulate
import itertools
import numpy as np
import os

def grid_simulation(drift_rates, noise_levels, verbose=False):
    # Setting up globals for simulations
    if verbose:
        print(f"Setting up grid simulation with drift rates \
{round(min(drift_rates), 2)}-{round(max(drift_rates), 2)} \
and noise_levels {round(min(noise_levels), 2)}-{round(max(noise_levels), 2)}")
    
    bin_size = 0.001
    seed = 42

    if verbose:
        print("Pulling empirical distributions")
    path = '/Users/braydenchien/Desktop/Enkavilab/DDM/1ms_trial_data.csv'
    empirical_distributions = get_empirical_distributions(path, bin_size)
        
    if verbose:
        print("Creating trials")
    num_trials = 500
    trials = create_trials(num_trials, empirical_distributions, seed=seed)

    # Generating model conditions from grid
    param_grid = np.array(list(itertools.product(drift_rates, noise_levels)))

    model_conditions_list = [
        {'drift_rate': round(pair[0], 2), 'theta': 0.38, 'noise': round(pair[1], 2)} for pair in param_grid
    ]

    if verbose:
        print("Starting simulations")
    # Simulating for each model conditions combination
    for model_conditions in model_conditions_list:
        if verbose:
            print(f"Attempting model_conditions {model_conditions}")
        try:
            simulate(bin_size, model_conditions, trials, seed=seed, save_results=True)
            if verbose:
                print(f"Model conditions {model_conditions} worked")
        except ValueError:
            if verbose:
                print(f'Model conditions {model_conditions} resulted in timeout')
            continue

        reformat_fixations(os.path.join('/Users/braydenchien/Desktop/Enkavilab/DDM', 'simulated_data'), f'sim_trials_s{seed}_d{model_conditions['drift_rate']}_t{model_conditions['theta']}_n{model_conditions['noise']}.csv')

        formatted_data_path = f'sim_trials_s{seed}_d{model_conditions['drift_rate']}_t{model_conditions['theta']}_n{model_conditions['noise']}.csv'
        save_fixation_properties('/Users/braydenchien/Desktop/Enkavilab/DDM/formatted_data', formatted_data_path)
        save_basic_psychometrics('/Users/braydenchien/Desktop/Enkavilab/DDM/formatted_data', formatted_data_path)

In [ ]:
drift_rates = np.arange(0.5, 2.6, 0.1)
noise_levels = np.arange(0.3, 0.7, 0.1)

grid_simulation(drift_rates, noise_levels, verbose=False)

In [ ]:
import seaborn as sns

def grid_simulation(drift_rates, noise_levels):
    theta = 0.38

    # Preallocate a grid for trial counts
    trial_grid = np.full((len(noise_levels), len(drift_rates)), np.nan)

    for i, noise in enumerate(noise_levels):
        for j, drift in enumerate(drift_rates):
            model = create_model(drift, theta, noise)
            try:
                results = simulate(bin_size, model, trials, seed, save_results=False) # Need to run create_trials and add result as parameter
                good_trials = valid_trials(results) # Need to create valid_trials function
                trial_grid[i, j] = good_trials

                if good_trials > len(trials) * 0.95:
                    print(f'Successful parameter combination drift rate: {drift}, noise: {noise}')

                    # Plot heatmap before returning
                    plt.figure(figsize=(8, 6))
                    sns.heatmap(trial_grid, annot=True, fmt=".0f",
                                xticklabels=np.round(drift_rates, 2),
                                yticklabels=np.round(noise_levels, 2),
                                cmap="YlGnBu")
                    plt.xlabel('Drift Rate')
                    plt.ylabel('Noise Level')
                    plt.title('Valid Trials per Parameter Combination')
                    plt.show()

                    return results

            except ValueError:
                print(f'Parameter combination drift rate: {drift}, noise: {noise} results in timeout')
                trial_grid[i, j] = np.nan  # Could also set to 0 if preferred

    # Plot heatmap even if no early return
    plt.figure(figsize=(8, 6))
    sns.heatmap(trial_grid, annot=True, fmt=".0f",
                xticklabels=np.round(drift_rates, 2),
                yticklabels=np.round(noise_levels, 2),
                cmap="YlGnBu")
    plt.xlabel('Drift Rate')
    plt.ylabel('Noise Level')
    plt.title('Valid Trials per Parameter Combination')
    plt.show()

    return None  # If no good result found

The above function needs a helper function `valid_trials` to assess the results of each simulation. The criteria are as follows:
- Two or more fixations in 495 of 500 trials
- Mean RT of around 1500 ms

In [ ]:
good_result_df = grid_simulation(np.linspace(0.5, 2.5, 5), np.linspace(0.3, 0.6, 4))